# 銷售分析｜營收成長與訂單趨勢

商業問題：  
月度營收趨勢與淡旺季特徵分析  
分析方法：  
- 依月份彙總營收
- 計算`三個月`移動平均，觀察營收趨勢
- 比較月度營收變化，辨識淡旺季

In [51]:
import pandas as pd
df=pd.read_csv("sales.csv")
df["Order_Date"]=pd.to_datetime(df["Order_Date"])
df=df.set_index("Order_Date")
monthly_sales=df["Total_Amount"].resample("ME").sum().rename("monthly_sales")
monthly_sales_3ma=monthly_sales.rolling(window=3).mean().rename("monthly_sales_3ma")
print(pd.concat([monthly_sales,monthly_sales_3ma],axis=1))

            monthly_sales  monthly_sales_3ma
Order_Date                                  
2024-06-30   2.407817e+08                NaN
2024-07-31   2.409663e+08                NaN
2024-08-31   2.401946e+08       2.406475e+08
2024-09-30   2.293313e+08       2.368307e+08
2024-10-31   2.399716e+08       2.364992e+08
2024-11-30   2.345578e+08       2.346202e+08
2024-12-31   2.419510e+08       2.388268e+08
2025-01-31   2.412292e+08       2.392460e+08
2025-02-28   2.151931e+08       2.327911e+08
2025-03-31   2.426311e+08       2.330178e+08
2025-04-30   2.414868e+08       2.331037e+08
2025-05-31   2.411965e+08       2.417715e+08
2025-06-30   2.322160e+08       2.382998e+08
2025-07-31   2.381340e+08       2.371822e+08
2025-08-31   2.461140e+08       2.388213e+08
2025-09-30   2.280220e+08       2.374233e+08
2025-10-31   2.474998e+08       2.405453e+08
2025-11-30   2.325721e+08       2.360313e+08
2025-12-31   2.436171e+08       2.412297e+08
2026-01-31   2.369286e+08       2.377059e+08
2026-02-28

分析結果：  
計算每月營收的 3 個月移動平均，用以平滑單月波動、觀察較平滑的營收趨勢。結果顯示營收結果長期維持在 2.33~2.42 億之間，波動幅度遠小於單月數據，未見明顯上升或下降趨勢，代表整體營收在此期間屬於平穩震盪，沒有結構性成長或衰退，單月的異常高低（如 2025 年 2 月低點、 3 月反彈）經平滑後已明顯被抹平。

商業問題：  
地區業績貢獻與營收集中度分析  
分析方法：  
- 依 `State` 彙總並排序各地區營收
- 計算`營收占比`與`累計占比`
- 評估地區業績集中程度與 80/20 法則

In [2]:
import pandas as pd
area=pd.read_csv("sales.csv")
State_Amount=area.groupby("State")["Total_Amount"].sum().rename("State_Amount").sort_values(ascending=False).reset_index()
Total_revenue=State_Amount["State_Amount"].sum()
State_Amount["Revenue_Share"]=State_Amount["State_Amount"]/Total_revenue
State_Amount["Cum_per"]=State_Amount["Revenue_Share"].cumsum()
print(State_Amount)

         State  State_Amount  Revenue_Share   Cum_per
0           UP  7.683510e+08       0.129555  0.129555
1    Rajasthan  7.572865e+08       0.127689  0.257244
2      Haryana  7.552152e+08       0.127340  0.384585
3  Maharashtra  6.058307e+08       0.102152  0.486736
4       Punjab  5.992254e+08       0.101038  0.587774
5      Gujarat  5.991777e+08       0.101030  0.688804
6        Delhi  4.744692e+08       0.080002  0.768807
7   Tamil Nadu  4.705157e+08       0.079336  0.848142
8  West Bengal  4.563200e+08       0.076942  0.925084
9    Karnataka  4.443018e+08       0.074916  1.000000


分析結果：  
在原本的邦別營收排名基礎上，新增「營收佔比」與「累計佔比」欄位。結果顯示業績貢獻排名前三大邦（ UP、Rajasthan、Haryana ）各佔約 13% ，並未達到 80/20 法則，顯示此印度電商的營收數相對均衡分布型態，未有「少數貢獻多數」由少數State獨占鰲頭的情形。

商業問題：  
取消訂單與退貨對實質營收的影響  
分析方法：  
- 篩選取消與退貨訂單  
- 分析各地區與付款方式的訂單數、占比與金額
- 找出取消與退貨較集中的地區及付款方式

In [2]:
import pandas as pd
from IPython.display import display
cancel=pd.read_csv("sales.csv")
cancel_filtered=cancel[cancel["Order_Status"].isin(["Cancelled","Returned"])]
ct_count=pd.crosstab(cancel_filtered["State"],cancel_filtered["Payment_Mode"],
                     margins=True,margins_name="Total").sort_values(by="COD",ascending=False)
ct_pct=round((pd.crosstab(cancel_filtered["State"],cancel_filtered["Payment_Mode"],
                          normalize="all")*100),2).sort_values(by="COD",ascending=False)
ct_amount=pd.crosstab(cancel_filtered["State"],cancel_filtered["Payment_Mode"],
                      values=cancel_filtered["Total_Amount"],aggfunc="sum").fillna(0).sort_values(by="COD",ascending=False)
print("取消與退貨訂單的數量統計：")
display(ct_count)
print("取消與退貨訂單的佔比統計：")
display(ct_pct)
print("取消與退貨訂單的金額統計：")
display(ct_amount)

取消與退貨訂單的數量統計：


Payment_Mode,COD,Credit Card,Debit Card,UPI,Total
State,,,,,
Total,8207,1070,2900,12823,25000
UP,1119,137,359,1690,3305
Haryana,1074,127,406,1664,3271
Rajasthan,997,162,348,1609,3116
Maharashtra,839,110,282,1315,2546
Gujarat,826,107,298,1273,2504
Punjab,818,110,299,1282,2509
Tamil Nadu,666,69,268,975,1978
West Bengal,646,87,227,1015,1975


取消與退貨訂單的佔比統計：


Payment_Mode,COD,Credit Card,Debit Card,UPI
State,,,,
UP,4.48,0.55,1.44,6.76
Haryana,4.30,0.51,1.62,6.66
Rajasthan,3.99,0.65,1.39,6.44
Maharashtra,3.36,0.44,1.13,5.26
Gujarat,3.30,0.43,1.19,5.09
Punjab,3.27,0.44,1.20,5.13
Tamil Nadu,2.66,0.28,1.07,3.90
West Bengal,2.58,0.35,0.91,4.06
Delhi,2.50,0.32,0.87,4.06


取消與退貨訂單的金額統計：


Payment_Mode,COD,Credit Card,Debit Card,UPI
State,,,,
UP,23927443.86,2559775.52,8477069.22,41891386.08
Rajasthan,22933163.27,5009908.71,7627187.89,40380688.97
Haryana,22812798.83,4211140.92,10099833.66,45897118.87
Maharashtra,19204838.90,3324857.12,5806573.34,32195730.24
Gujarat,18694781.61,2684803.58,7311407.52,31684597.11
Punjab,17774196.84,3240097.47,6690369.85,33641941.70
Tamil Nadu,17058506.17,2297321.12,5187371.47,23996811.10
West Bengal,14996261.28,1815450.49,5408924.26,23824340.43
Karnataka,13953242.50,1605245.41,4693219.55,21147017.65


分析結果：  
進一步交叉分析取消與退貨訂單在邦別與付款方式上的分佈（數量、佔比、金額）。結果顯示，取消與退貨訂單主要集中於營收較高的邦，付款方式則以 UPI 與 COD 為主，可進一步比較各地區及付款方式的取消／退貨率，以評估實際風險。

商業問題：  
付款方式與物流成本結構分析  
分析方法：  
- 依 `Payment_Mode` 彙總訂單金額與物流成本
- 計算各付款方式的物流成本占比
- 依物流成本占比排序，比較不同付款方式的成本結構

In [26]:
import pandas as pd
pm=pd.read_csv("sales.csv")
result=pm.groupby("Payment_Mode")[["Order_Value","Shipping_Cost"]].sum().reset_index()
result["per"]=(result["Shipping_Cost"]/result["Order_Value"])*100
result=result.sort_values(by="per",ascending=False)
print(result)

  Payment_Mode   Order_Value  Shipping_Cost       per
2   Debit Card  6.384033e+08      152112.04  0.023827
0          COD  1.860675e+09      423972.83  0.022786
3          UPI  3.209293e+09      605384.59  0.018863
1  Credit Card  2.835934e+08       46523.36  0.016405


分析結果：  
計算各付款方式的物流成本佔比，並依佔比高低排序。結果顯示 Debit Card 物流成本佔比最高（約 0.0238% ）， Credit Card 最低（ 約0.0164% ），客單價較低的付款方式，其物流成本佔比相對較高。不過整體差異僅在小數點後幾位，顯示不同付款方式的物流成本結構差異有限。
